In [5]:
import pandas as pd
import numpy as np
import re
import json
import os
from tqdm import tqdm
from pathlib import Path

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from statsmodels.stats.proportion import proportions_ztest

In [6]:
df = pd.read_csv("linkedin_job_postings_after_eda.csv")

print("Loaded dataset shape:", df.shape)
print(df.columns.tolist())

df.head()

Loaded dataset shape: (36256, 22)
['company_name', 'title', 'description', 'location', 'formatted_work_type', 'formatted_experience_level', 'skills_desc', 'normalized_salary', 'exp_level_3', 'salary_signal', 'skills_desc_clean', 'skill_count', 'skill_count_band', 'advanced_skill_count', 'mid_skill_count', 'skill_signal', 'skill_signal_num', 'title_signal', 'title_signal_num', 'years_experience', 'years_signal', 'years_signal_num']


,company_name,title,description,location,formatted_work_type,formatted_experience_level,skills_desc,normalized_salary,exp_level_3,salary_signal,...,skill_count_band,advanced_skill_count,mid_skill_count,skill_signal,skill_signal_num,title_signal,title_signal_num,years_experience,years_signal,years_signal_num
0,Galerie Candy and Gifts,Quality Assurance Manager,Galerie is seeking an experienced Quality Assu...,"Hebron, KY",Full-time,Mid-Senior level,NaN,NaN,mid,unknown,...,unknown,0,0,unknown,0,senior,3,0,unknown,0
1,Webologix Ltd/ INC,Anaplan Developer,Job Title: Anaplan Developer\n\nLocations: US ...,United States,Full-time,Mid-Senior level,NaN,NaN,mid,unknown,...,unknown,0,0,unknown,0,mid,2,0,unknown,0
2,Crisp,Account Executive - Mid-Market,"Here at Crisp, we value the strength in teamwo...","Chicago, IL",Full-time,Mid-Senior level,NaN,NaN,mid,unknown,...,unknown,0,0,unknown,0,mid,2,0,unknown,0
3,USLI,Senior Developer,This individual will work with a high performa...,Greater Philadelphia,Full-time,Associate,NaN,NaN,junior,unknown,...,unknown,0,0,unknown,0,senior,3,10,senior,3
4,TECHEAD,Inbound Call Center Specialist,"Always Connecting, Always Evolving.\nIf you ar...","Richmond, VA",Contract,Associate,NaN,38480.0,junior,junior,...,unknown,0,0,unknown,0,mid,2,30,senior,3


In [7]:
# ============================================
# SCRIPT 1: DATA_PREPARATION.py
# Run this ONCE on your main system
# ============================================

import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

# Load and preprocess data
df = pd.read_csv("linkedin_job_postings_after_eda.csv")
print(f"Loaded dataset shape: {df.shape}")

# Map experience level
def map_experience_level(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if any(term in x for term in ["intern", "entry", "junior", "associate", "trainee"]):
        return "junior"
    elif any(term in x for term in ["mid", "intermediate"]):
        return "mid"
    elif any(term in x for term in ["senior", "lead", "manager", "director", "executive", "principal", "head", "vp"]):
        return "senior"
    else:
        return np.nan

df["experience_level"] = df["formatted_experience_level"].apply(map_experience_level)
df = df.dropna(subset=["experience_level"]).reset_index(drop=True)
print(f"Rows after target cleaning: {len(df)}")

# Feature engineering (salary signal)
def salary_signal(x):
    if pd.isna(x):
        return "unknown"
    try:
        x = float(x)
    except:
        return "unknown"
    if x < 50000:
        return "junior"
    elif x <= 100000:
        return "mid"
    else:
        return "senior"

df["salary_signal"] = df["normalized_salary"].apply(salary_signal)

# Feature engineering (skills)
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9,+/#.\-\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["skills_text"] = df["skills_desc"].apply(clean_text)

def count_skills(text):
    if not text:
        return 0
    parts = re.split(r",|;|\||\n", text)
    parts = [p.strip() for p in parts if p.strip()]
    parts = list(dict.fromkeys(parts))
    return len(parts)

df["skill_count"] = df["skills_text"].apply(count_skills)

advanced_skills = [
    "machine learning", "deep learning", "nlp", "computer vision",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform",
    "spark", "hadoop", "airflow", "pytorch", "tensorflow",
    "mlops", "big data", "data engineering", "architecture",
    "microservices", "llm", "genai"
]

mid_skills = [
    "python", "sql", "java", "excel", "tableau", "power bi",
    "statistics", "data analysis", "git", "linux", "etl",
    "scikit-learn", "pandas", "numpy", "matplotlib"
]

def keyword_count(text, keywords):
    text = str(text).lower()
    return sum(1 for kw in keywords if kw in text)

df["advanced_skill_count"] = df["skills_text"].apply(lambda x: keyword_count(x, advanced_skills))
df["mid_skill_count"] = df["skills_text"].apply(lambda x: keyword_count(x, mid_skills))

def skill_signal(row):
    adv = row["advanced_skill_count"]
    midc = row["mid_skill_count"]
    total = row["skill_count"]
    if adv >= 2 or total >= 8:
        return "senior"
    elif adv == 1 or midc >= 2 or 4 <= total <= 7:
        return "mid"
    elif 1 <= total <= 3:
        return "junior"
    else:
        return "unknown"

df["skill_signal"] = df.apply(skill_signal, axis=1)

def title_signal(title):
    title = str(title).lower()
    if any(w in title for w in ["intern", "trainee", "junior", "associate"]):
        return "junior"
    elif any(w in title for w in ["senior", "lead", "principal", "manager", "director", "head"]):
        return "senior"
    else:
        return "mid"

df["title_signal"] = df["title"].apply(title_signal)

def extract_years(text):
    text = str(text).lower()
    matches = re.findall(r"(\d+)\+?\s*(?:years|year|yrs|yr)", text)
    if matches:
        return max(int(m) for m in matches)
    return 0

df["years_experience"] = df["description"].apply(extract_years)

def years_signal(y):
    if y == 0:
        return "unknown"
    elif y <= 2:
        return "junior"
    elif y <= 5:
        return "mid"
    else:
        return "senior"

df["years_signal"] = df["years_experience"].apply(years_signal)

# Create combined text
df["combined_text"] = (
    df["title"].astype(str).str.strip() + " | " +
    df["description"].astype(str).str.strip()
)

# ============================================
# SPLIT INTO 13 CHUNKS
# ============================================
NUM_CHUNKS = 13
TOTAL_ROWS = len(df)

# Stratified split by experience_level to maintain distribution
chunks = []
for level in df["experience_level"].unique():
    level_df = df[df["experience_level"] == level].reset_index(drop=True)
    level_size = len(level_df)
    rows_per_chunk = level_size // NUM_CHUNKS

    for i in range(NUM_CHUNKS):
        start = i * rows_per_chunk
        if i == NUM_CHUNKS - 1:
            end = level_size  # Last chunk gets remainder
        else:
            end = (i + 1) * rows_per_chunk
        chunks.append(level_df.iloc[start:end])

# Combine chunks by chunk index
final_chunks = []
for i in range(NUM_CHUNKS):
    chunk_df = pd.concat([chunks[j * NUM_CHUNKS + i] for j in range(len(df["experience_level"].unique()))])
    chunk_df = chunk_df.reset_index(drop=True)
    final_chunks.append(chunk_df)

# Save chunks
output_dir = Path("job_classification_chunks")
output_dir.mkdir(exist_ok=True)

for i, chunk in enumerate(final_chunks):
    chunk_path = output_dir / f"chunk_{i+1:02d}.csv"
    chunk.to_csv(chunk_path, index=False)
    print(f"Saved chunk {i+1}/{NUM_CHUNKS}: {len(chunk)} rows -> {chunk_path}")

# Save metadata
metadata = {
    "total_rows": TOTAL_ROWS,
    "num_chunks": NUM_CHUNKS,
    "chunk_sizes": [len(c) for c in final_chunks]
}
import json
with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Data preparation complete!")
print(f"Total rows: {TOTAL_ROWS}")
print(f"Number of chunks: {NUM_CHUNKS}")
print(f"Chunk directory: {output_dir.absolute()}")

Loaded dataset shape: (36256, 22)
Rows after target cleaning: 36256
Saved chunk 1/13: 2787 rows -> job_classification_chunks/chunk_01.csv
Saved chunk 2/13: 2787 rows -> job_classification_chunks/chunk_02.csv
Saved chunk 3/13: 2787 rows -> job_classification_chunks/chunk_03.csv
Saved chunk 4/13: 2787 rows -> job_classification_chunks/chunk_04.csv
Saved chunk 5/13: 2787 rows -> job_classification_chunks/chunk_05.csv
Saved chunk 6/13: 2787 rows -> job_classification_chunks/chunk_06.csv
Saved chunk 7/13: 2787 rows -> job_classification_chunks/chunk_07.csv
Saved chunk 8/13: 2787 rows -> job_classification_chunks/chunk_08.csv
Saved chunk 9/13: 2787 rows -> job_classification_chunks/chunk_09.csv
Saved chunk 10/13: 2787 rows -> job_classification_chunks/chunk_10.csv
Saved chunk 11/13: 2787 rows -> job_classification_chunks/chunk_11.csv
Saved chunk 12/13: 2787 rows -> job_classification_chunks/chunk_12.csv
Saved chunk 13/13: 2812 rows -> job_classification_chunks/chunk_13.csv

✅ Data preparatio

In [8]:
!nvcc --version

!wget https://antidote.cloud/f/ae5312aa983845c7abf1/?dl=1 -O llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl

#!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama_cpp_python
!CMAKE_ARGS="-DGGML_CUDA=on" pip install /content/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
!pip install huggingface_hub hf_transfer
%env HF_HUB_ENABLE_HF_TRANSFER=1
!huggingface-cli download bartowski/Mistral-Nemo-Instruct-2407-GGUF Mistral-Nemo-Instruct-2407-Q5_K_M.gguf --local-dir . --local-dir-use-symlinks False

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
--2026-03-12 14:53:05--  https://antidote.cloud/f/ae5312aa983845c7abf1/?dl=1
Resolving antidote.cloud (antidote.cloud)... 193.30.122.219
Connecting to antidote.cloud (antidote.cloud)|193.30.122.219|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://antidote.cloud/seafhttp/files/d4ded823-5da3-4ac1-906c-6c76d8c540d9/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl [following]
--2026-03-12 14:53:06--  https://antidote.cloud/seafhttp/files/d4ded823-5da3-4ac1-906c-6c76d8c540d9/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
Reusing existing connection to antidote.cloud:443.
HTTP request sent, awaiting response... 200 OK
Length: 52058581 (50M) [application/octet-stream]
Saving to: ‘llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl’

llama_c

In [9]:
# ------------------------------------------------------------
# INSTALL LLAMA-CPP-PYTHON (PRE-BUILT CUDA WHEEL)
# ------------------------------------------------------------
print("="*60)
print("STEP 2: INSTALLING LLAMA-CPP-PYTHON")
print("="*60)

# Uninstall any existing versions to avoid conflicts
!pip uninstall -y llama_cpp_python llama-cpp-python 2>/dev/null || true

# Install the pre-built wheel you already have (NO CMAKE_ARGS!)
!pip install -q ./llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl

print("\n✅ Installation complete")

STEP 2: INSTALLING LLAMA-CPP-PYTHON
Found existing installation: llama_cpp_python 0.3.16
Uninstalling llama_cpp_python-0.3.16:
  Successfully uninstalled llama_cpp_python-0.3.16

✅ Installation complete


In [10]:
# ------------------------------------------------------------
# DOWNLOAD MODEL WITH FULL VERIFICATION
# ------------------------------------------------------------
import os
import time

print("="*60)
print("STEP 1: DOWNLOADING MISTRAL NEMO MODEL")
print("="*60)

# Install required packages
!pip install -q huggingface_hub hf_transfer
%env HF_HUB_ENABLE_HF_TRANSFER=1

# Attempt download with retries
model_filename = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
max_retries = 3

for attempt in range(max_retries):
    print(f"\nAttempt {attempt+1}/{max_retries}...")
    try:
        !huggingface-cli download bartowski/Mistral-Nemo-Instruct-2407-GGUF {model_filename} --local-dir . --local-dir-use-symlinks False

        # Check if file exists
        if os.path.exists(model_filename):
            size_gb = os.path.getsize(model_filename) / (1024**3)
            print(f"\n✅ Download successful!")
            print(f"   Filename: {model_filename}")
            print(f"   Size: {size_gb:.2f} GB")
            MODEL_PATH = os.path.abspath(model_filename)
            break
        else:
            print("⚠️ File not found after download. Retrying...")
            time.sleep(3)
    except Exception as e:
        print(f"⚠️ Error: {str(e)[:100]}")
        time.sleep(3)
else:
    # Fallback to manual wget download
    print("\n❌ huggingface-cli failed. Trying manual wget download...")
    !wget -q --show-progress https://huggingface.co/bartowski/Mistral-Nemo-Instruct-2407-GGUF/resolve/main/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf -O {model_filename}

    if os.path.exists(model_filename):
        size_gb = os.path.getsize(model_filename) / (1024**3)
        print(f"\n✅ Manual download succeeded!")
        print(f"   Size: {size_gb:.2f} GB")
        MODEL_PATH = os.path.abspath(model_filename)
    else:
        raise FileNotFoundError("❌ Model download failed completely. Check disk space with !df -h")

print(f"\n✓ MODEL_PATH defined: {MODEL_PATH}")

STEP 1: DOWNLOADING MISTRAL NEMO MODEL
env: HF_HUB_ENABLE_HF_TRANSFER=1

Attempt 1/3...
/bin/bash: line 1: huggingface-cli: command not found
⚠️ File not found after download. Retrying...

Attempt 2/3...
/bin/bash: line 1: huggingface-cli: command not found
⚠️ File not found after download. Retrying...

Attempt 3/3...
/bin/bash: line 1: huggingface-cli: command not found
⚠️ File not found after download. Retrying...

❌ huggingface-cli failed. Trying manual wget download...
Mistral-Nemo-Instru 100%[===================>]   8.13G   130MB/s    in 96s     

✅ Manual download succeeded!
   Size: 8.13 GB

✓ MODEL_PATH defined: /content/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf


In [11]:
from llama_cpp import Llama

MODEL_PATH = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=1024,
n_gpu_layers =35 , # All layers on GPU
n_threads =8 , # CPU threads
n_batch =512 , # Batch size
use_mmap=True ,
use_mlock=False ,
seed = 1234
)

print("Mistral model loaded successfully.")

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) - 14807 MiB free
llama_model_loader: loaded meta data with 44 key-value pairs and 363 tensors from Mistral-Nemo-Instruct-2407-Q5_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Mistral Nemo Instruct 2407
llama_model_loader: - kv   3:                            general.version str              = 2407
llama_model_loader: - kv   4:                           general.finetune str       

Mistral model loaded successfully.


CUDA : ARCHS = 750 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'general.quantization_version': '2', 'tokenizer.chat_template': '{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = messages[0]["content"] %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set loop_messages = messages %}\n{%- endif %}\n{%- if not tools is defined %}\n    {%- set tools = none %}\n{%- endif %}\n{%- set user_messages = loop_messages | selectattr("role", "equalto", "user") | list %}\n\n{#- This block checks for alternating user/assistant messages, skipping tool calling messages #}\n{%- set ns = namespace() %}\n{%- set ns.index = 0 %}\n{%- for message in loop_messages %}\n    {%- if not (message.role == "tool" or message.role == "tool_results" or (message.tool_calls is defined and message.tool_calls is not none)) %}\n        {%- 

In [12]:
# ============================================
# SCRIPT 2: WORKER_INFERENCE.py
# Run this on EACH of the 13 systems
# Modify CHUNK_ID for each system (1-13)
# ============================================

import pandas as pd
import numpy as np
import re
import json
import os
from pathlib import Path
from tqdm import tqdm
from llama_cpp import Llama

# ============================================
# CONFIGURATION - CHANGE THIS FOR EACH SYSTEM
# ============================================
CHUNK_ID = 9  # Change this: 1, 2, 3, ... 13
MODEL_PATH = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
INPUT_CHUNK = f"job_classification_chunks/chunk_{CHUNK_ID:02d}.csv"
OUTPUT_FILE = f"job_classification_chunks/results_chunk_{CHUNK_ID:02d}.csv"
CHECKPOINT_FILE = f"job_classification_chunks/checkpoint_chunk_{CHUNK_ID:02d}.json"

# ============================================
# LOAD DATA
# ============================================
print(f"Loading chunk {CHUNK_ID}...")
df = pd.read_csv(INPUT_CHUNK)
print(f"Loaded {len(df)} rows")


# ============================================
# PROMPT BUILDING
# ============================================
def build_prompt(row):
    title = str(row.get("title", ""))
    description = str(row.get("description", ""))[:3000]
    salary_signal = row.get("salary_signal", "unknown")
    skill_signal = row.get("skill_signal", "unknown")
    title_signal_val = row.get("title_signal", "unknown")
    years_signal_val = row.get("years_signal", "unknown")
    skill_count = row.get("skill_count", 0)

    prompt = f"""You are classifying a job posting into exactly one of these labels:
junior, mid, senior

Instructions:
- Use the title and description as the main evidence.
- Use the engineered hints only as supporting evidence.
- Return ONLY valid JSON in this exact format:
{{"label": "junior"}}
or
{{"label": "mid"}}
or
{{"label": "senior"}}

Job title:
{title}

Job description:
{description}

Supporting hints:
- salary_signal: {salary_signal}
- skill_signal: {skill_signal}
- title_signal: {title_signal_val}
- years_signal: {years_signal_val}
- skill_count: {skill_count}
"""
    return prompt.strip()

# ============================================
# LABEL EXTRACTION
# ============================================
VALID_LABELS = {"junior", "mid", "senior"}

def extract_label(output_text):
    text = str(output_text).strip().lower()
    try:
        parsed = json.loads(text)
        label = parsed.get("label", "").strip().lower()
        if label in VALID_LABELS:
            return label
    except:
        pass
    for label in ["junior", "mid", "senior"]:
        if re.search(rf'\b{label}\b', text):
            return label
    return "unknown"

# ============================================
# CHECK FOR RESUME POINT
# ============================================
start_row = 0
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        checkpoint = json.load(f)
        start_row = checkpoint.get("last_processed_row", 0)
        print(f"Resuming from row {start_row}")

# ============================================
# RUN INFERENCE
# ============================================
preds = []
raw_outputs = []

for i in tqdm(range(start_row, len(df)), desc=f"Chunk {CHUNK_ID}"):
    row = df.iloc[i]
    prompt = build_prompt(row)

    try:
        response = llm(
            prompt,
            max_tokens=20,
            temperature=0.0,
            top_p=1.0,
            echo=False
        )
        raw_output = response["choices"][0]["text"]
        pred = extract_label(raw_output)
    except Exception as e:
        raw_output = f"ERROR: {str(e)}"
        pred = "unknown"

    preds.append(pred)
    raw_outputs.append(raw_output)

    # Save checkpoint every 10 rows
    if (i + 1) % 10 == 0:
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump({"last_processed_row": i + 1}, f)

# ============================================
# SAVE RESULTS
# ============================================
df["predicted_level"] = preds
df["raw_output"] = raw_outputs
df.to_csv(OUTPUT_FILE, index=False)

# Clear checkpoint
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

print(f"\n✅ Chunk {CHUNK_ID} complete!")
print(f"Saved: {OUTPUT_FILE}")
print(f"Rows processed: {len(df)}")

Loading chunk 9...
Loaded 2787 rows


Streaming output truncated to the last 5000 lines.
Chunk 9:  70%|███████   | 1954/2787 [3:09:54<1:10:08,  5.05s/it]Llama.generate: 83 prefix-match hit, remaining 571 prompt tokens to eval
llama_perf_context_print:        load time =    2225.01 ms
llama_perf_context_print: prompt eval time =    1799.32 ms /   571 tokens (    3.15 ms per token,   317.34 tokens per second)
llama_perf_context_print:        eval time =    4367.31 ms /    19 runs   (  229.86 ms per token,     4.35 tokens per second)
llama_perf_context_print:       total time =    6194.82 ms /   590 tokens
llama_perf_context_print:    graphs reused =         17
Chunk 9:  70%|███████   | 1955/2787 [3:10:00<1:14:53,  5.40s/it]Llama.generate: 83 prefix-match hit, remaining 639 prompt tokens to eval
llama_perf_context_print:        load time =    2225.01 ms
llama_perf_context_print: prompt eval time =    1784.65 ms /   639 tokens (    2.79 ms per token,   358.05 tokens per second)
llama_perf_context_print:        eval time =    5


✅ Chunk 9 complete!
Saved: job_classification_chunks/results_chunk_09.csv
Rows processed: 2787
